# Technical Assessment FamilyMart Internship
<br>

Name : Stewart Tang Jia Heng <br>
Task 1 - Car Counter <br>

Build a solution to count the number of cars passing through a
given point using video footage.


Installing necessary packages needed for this tasks.

In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.3 MB/s eta 0:00:00


In [5]:
!git clone https://github.com/abewley/sort.git

Cloning into 'sort'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 208 (delta 45), reused 40 (delta 40), pack-reused 159 (from 1)
Receiving objects: 100% (208/208), 1.20 MiB | 20.20 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [6]:
%cd sort

/content/sort


In [7]:
!pip install filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=fbf607beed9e7af290293fda50d54272d603e518b23c40d16d2754684e153570
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


Importing necessary packages needed for the task.

In [143]:
from ultralytics import YOLO
from sort import Sort
from google.colab.patches import cv2_imshow
import cv2
import numpy as np
import time

Initialisation of the model, video and tracker.

In [153]:
def initialisation(video_link):
  model = YOLO("yolov8n.pt")
  video = cv2.VideoCapture(video_link)
  tracker = Sort()
  return video, model, tracker

Method for detecting the vehicles in the videos with setting a confidence score to identify how confidence is that frame is.

In [62]:
def detect(model, frame, car_types, conf_score = 0.5):
  car_detection = model(frame)[0]
  detections = []
  for detection in car_detection.boxes.data.tolist():
    x1, y1, x2, y2, c_score, class_id = detection
    if class_id in car_types and c_score >= conf_score:
      detections.append([x1, y1, x2, y2, c_score])
  return np.array(detections)

Tracking the speed of the vehicles of the video and calculating the speed in km/h.

In [154]:
def track_speed(tracker, detections, prev_pos, speed, fps, pixel_to_meter):
  if detections is None or len(detections) == 0:
        return [], prev_pos, speed
  detections = detections.astype(np.float32)
  car_tracker = tracker.update(detections)
  for car in car_tracker:
    x1, y1, x2, y2, class_id = map(int, car)
    center_x = int((x1 + x2)/2)
    center_y = int((y1 + y2)/2)
    if class_id in prev_pos:
      pos_x, pos_y = prev_pos[class_id]
      distance = np.sqrt((center_x - pos_x)**2 + (center_y - pos_y)**2)
      distance_in_meter = distance * pixel_to_meter
      car_speed = distance_in_meter * fps * 3.6
      speed[class_id] = car_speed
    prev_pos[class_id] = (center_x, center_y)
  return car_tracker, prev_pos, speed


Showing the visualisation of the needed information in the output video for example the car counts and the box for selecting vehicles.

In [87]:
def visualise(frame, car_tracker, speed, line_position, car_count):
  for car in car_tracker:
    color = (0, 255, 0)
    x1, y1, x2, y2, class_id = map(int, car)
    cv2.rectangle(frame, (x1 , y1),(x2 , y2), color, 2)
    text = f"ID {class_id}: {speed.get(class_id, 0):.1f} km/h"
    cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)
    cv2.putText(frame, f"Cars: {car_count}", (50, 50),
                cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255), 2)
    return frame

Processing the video by using the model to detect the car counts in the video and also included different classes of vehicles for example motorbike bus and truck not specifying only car.

In [151]:
def process_video(video_link, output_path = "/content/output.mp4"):
  car_types = [2, 3, 5, 7] #class id for car, motorbike, bus and truck
  car_count = 0
  line_position = 500
  pixel_to_meter = 0.02

  video, model, tracker = initialisation(video_link)
  fps = int(video.get(cv2.CAP_PROP_FPS))
  width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

  fourcc = cv2.VideoWriter_fourcc(*'mp4v')
  out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

  prev_pos = {}
  speed = {}
  counted_ids = set()

  while True:
    ret, frame = video.read()
    if not ret:
      break
    detections = detect(model, frame, car_types)
    cv2.line(frame, (line_position, 0), (line_position, height), (0, 0, 255), 2)
    car_tracker, prev_pos, speed = track_speed(tracker, detections, prev_pos, speed, fps, pixel_to_meter)
    for car in car_tracker:
      x1, y1, x2, y2, obj_id = map(int, car)
      center_x = int((x1 + x2) / 2)
      if abs(center_x - line_position) < 10 and obj_id not in counted_ids:
        car_count += 1
        counted_ids.add(obj_id)
        print(f"✅ Car {obj_id} crossed line. Total: {car_count}")
    frame = visualise(frame, car_tracker, speed, line_position, car_count)
    out.write(frame)

  video.release()
  out.release()
  print(f"\nTotal car count:{car_count}\n")
  return output_path

Outputting the video.

In [152]:
process_video("/content/Car_Count.mp4")


0: 384x640 1 car, 10.4ms
Speed: 1.6ms preprocess, 10.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 6.9ms
Speed: 1.6ms preprocess, 6.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 6.8ms
Speed: 1.4ms preprocess, 6.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 7.3ms
Speed: 1.5ms preprocess, 7.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 7.4ms
Speed: 0.9ms preprocess, 7.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 7.5ms
Speed: 1.1ms preprocess, 7.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 6.6ms
Speed: 1.4ms preprocess, 6.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 7.8ms
Speed: 1.4ms preprocess, 7.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 6.8ms
Speed

'/content/output.mp4'